# 63 — Run Blind-A with config 170 (wRRF + SID) → submission zip

ONLY run this after notebook 64's dev gate has PASSED. This produces the
prediction.json for CodaBench. Wallclock ~60-100 min on L4.


In [ ]:
# 1) Setup (same pattern as notebook 62).
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('sid', 'recsys2026_sid_cache'),
    ('dense', 'recsys2026_dense_cache'),
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

!pip install -q -U "peft>=0.10" "transformers>=4.40" "accelerate>=0.30" "torchao>=0.17"


In [ ]:
# 2) Run config 170 on Blind-A.
%cd /content/recsys2026/music-crs-baselines
!python run_inference_blindset.py \
    --tid 170-wrrf-sid-v5kto-blindsetA \
    --batch_size 32 \
    2>&1 | tee /content/drive/MyDrive/recsys2026_sid_generator_cache/blindA_170_log.txt | tail -60

In [ ]:
# 3) Validate the prediction.json: must be 80 entries (80 unique sessions × 1 turn each).
%cd /content/recsys2026
import json
pred_path = 'music-crs-baselines/exp/inference/blindset_A/170-wrrf-sid-v5kto-blindsetA.json'
preds = json.load(open(pred_path))
n_entries = len(preds) if isinstance(preds, list) else len(preds.keys())
print(f'prediction file: {n_entries} entries')
assert n_entries == 80, f'EXPECTED 80, got {n_entries} — DO NOT submit'
sample_keys = list(preds[0].keys()) if isinstance(preds, list) else list(list(preds.values())[0].keys())
print(f'sample entry keys: {sample_keys}')

In [ ]:
# 4) Validate per existing validator (catches schema bugs before submission).
# Use subprocess + assert rc==0 so a failing precheck or validator HALTS the cell
# (IPython's `!` doesn't propagate non-zero exit codes — bad payloads would slip
# through to the zip step otherwise).
%cd /content/recsys2026
import subprocess
import sys
sys.path.insert(0, '/content/recsys2026')
from scripts.precheck_prediction import precheck, _load_catalog
from pathlib import Path

PRED_PATH = Path('music-crs-baselines/exp/inference/blindset_A/170-wrrf-sid-v5kto-blindsetA.json')

# Stricter precheck — call directly (avoids subprocess + redundant catalog load).
catalog = _load_catalog("talkpl-ai/TalkPlayData-Challenge-Track-Metadata")
result = precheck(PRED_PATH, catalog=catalog, expected_n=80)
assert result['ok'], (
    f'precheck FAILED with {len(result["errors"])} errors; '
    f'first 5: {result["errors"][:5]}'
)
print(f'OK: precheck passed (n_records={result["n_records"]})')

# Schema validator (less strict but covers different bugs).
rc = subprocess.call(['python', 'scripts/validate_prediction.py', '--input', str(PRED_PATH)])
assert rc == 0, f'validate_prediction.py FAILED (rc={rc}) — do not submit'
print('OK: schema validator passed')

In [ ]:
# 5) Zip for CodaBench (prediction.json must be at the ROOT of the zip).
import os, zipfile, datetime
date_str = datetime.date.today().strftime('%Y-%m-%d')
zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{date_str}-sid-ensemble-170.zip'
os.makedirs(os.path.dirname(zip_path), exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname='prediction.json')
print(f'zip ready: {zip_path}')
print(f'size: {os.path.getsize(zip_path) / 1024:.1f} KB')

## After the run

1. Download the zip from Drive: `/content/drive/MyDrive/recsys2026_submissions/<date>-sid-ensemble-170.zip`
2. Upload to CodaBench (https://www.codabench.org/competitions/...).
3. Record the composite + nDCG@20 + LLM + lex_div scores in the project memory under a new `project_blind_a_w4_first_submission.md`.
4. If results are positive: proceed to W5 weight tuning + config 171 (pure-SID) comparison.
5. If results are negative: debug — likely the SID weight (0.5) is too high or the SID generator isn't strong enough; revisit.
